# Monitor and audit remote interface workflows

Track MLIP/VASP jobs by their root UUID, then audit stored interface structures and compressed VASP products. Replace the example UUIDs with values printed by the submission notebooks.

In [ ]:
from pathlib import Path
import sys

import InterOptimus

candidates = (Path.cwd(), Path.cwd().parent, Path(InterOptimus.__file__).resolve().parents[1])
REPO_ROOT = next((path for path in candidates if (path / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the InterOptimus repository or its examples directory")
sys.path.insert(0, str(REPO_ROOT))

from InterOptimus.agents.remote_submit import (
    iomaker_fetch_results,
    iomaker_status,
    query_interoptimus_task_progress,
)
from scripts.audit_interface_workflow_results import audit_mlip, audit_vasp

# Fill these from the `mlip_job_uuid` values printed after submission.
MLIP_JOBS = {
    # "orb_si_ge": "00000000-0000-0000-0000-000000000000",
}
VASP_JOBS = {
    # "standard_si_ge": "00000000-0000-0000-0000-000000000000",
}

for name, uuid in {**MLIP_JOBS, **VASP_JOBS}.items():
    print(f"\n{name}")
    iomaker_status(uuid)

In [ ]:
audit_report = {
    "mlip": [audit_mlip(name, uuid) for name, uuid in MLIP_JOBS.items()],
    "vasp": [audit_vasp(name, uuid) for name, uuid in VASP_JOBS.items()],
}

for item in audit_report["mlip"]:
    print("MLIP", item["name"], "invariants:", item.get("all_invariants_pass"))
for item in audit_report["vasp"]:
    print("VASP", item["name"], "products:", item.get("all_products_complete"))

audit_report

In [ ]:
# Fetch selected completed results into a local export directory when needed.
FETCH = False
if FETCH:
    fetched = {
        name: iomaker_fetch_results(uuid)
        for name, uuid in {**MLIP_JOBS, **VASP_JOBS}.items()
    }
    fetched
else:
    print("Set FETCH = True to download result bundles.")